In [ ]:
import subprocess
import sys
import os

kr_pairs = [
    # r = 2
    (2, 2), (3, 2), (4, 2), (5, 2), (6, 2),
    (7, 2), (8, 2), (9, 2), (10, 2), (11, 2), (12, 2),

    # r = 3
    (2, 3), (3, 3), (4, 3), (5, 3),

    # r = 4
    (2, 4), (3, 4),

    # r = 5
    (2, 5),

    # r = 6
    (2, 6),
]

def sffa_budget(k, r):
    return sum(k ** ell for ell in range(1, r + 1))

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

for k, r in kr_pairs:
    budget = sffa_budget(k, r)

    print("\n" + "=" * 80, flush=True)
    print(f"Budget analysis: p=0.5, k={k}, r={r}, budget={budget}", flush=True)
    print("=" * 80, flush=True)

    cmd = [
        sys.executable, "-u", "sfl_reconstruction.py",
        "--device", "cuda",
        "--num_runs", "10",
        "--optimizer", "lbfgs",
        "--lr", "3e-2",
        "--epochs", "20",
        "--lbfgs_max_iter", "50",

        "--p", "0.5",
        "--p1", "0.1",

        "--k", str(k),
        "--r", str(r),

        "--methods", "pure_sffa",
        "--max_sffa_basis_size", "1000",

        "--ridge", "1e-6",
        "--spectral_preprocess", "topk",
        "--bandlimit_k", "250",
        "--split_mode", "number",
        "--train_size", "200",
        "--test_size", "200",
        "--data_dir", "windmill_processed",
        "--adjacency_file", "A.npy",

        "--no-save_outputs",
    ]

    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )

    for line in proc.stdout:
        print(line, end="", flush=True)

    return_code = proc.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, cmd)